In [0]:
## EDA

Overview - Dimensional_Product table

In [0]:
%sql

select *
from ecommerce.gold.gld_products
limit 5

In [0]:
%sql
select max('product_id') as Attributes ,count(product_id) as count_value from ecommerce.gold.gld_products
union all
select max('Total_Sku') as Attributes ,count(distinct sku) as count_value from ecommerce.gold.gld_products
union all
select max('Total_unique_brand') as Attributes ,count(distinct brand_code) as count_value from ecommerce.gold.gld_products
union all
select max('Total_unique_category') as Attributes ,count(distinct category_code) as count_value from ecommerce.gold.gld_products
union all
select max('unique_product_color') as Attributes ,count(distinct color) as count_value from ecommerce.gold.gld_products
union all
select max('distinct_size') as Attributes ,count(distinct size) as count_value from ecommerce.gold.gld_products
union all
select max('material_used') as Attributes ,count(distinct material) as count_value from ecommerce.gold.gld_products



Dimensional Customer table overview

In [0]:
%sql 
select max('Total_registered_customers')as attributes ,count(*) as count_value from ecommerce.gold.gld_customers
union all
select max('Total_unique_country')as attributes ,count(distinct country) as count_value from ecommerce.gold.gld_customers
union all
select max('Total_unique_state')as attributes ,count(distinct state_code) as count_value from ecommerce.gold.gld_customers
union all
select max('Total_unique_region')as attributes ,count(distinct regions) as count_value from ecommerce.gold.gld_customers


Fact Order_items table overview

In [0]:
%sql
select *
from ecommerce.gold.gld_order_items
limit 5;

Fact_table: KPI's

In [0]:
%sql
select *  from ecommerce.gold.gld_order_items limit 5;

In [0]:
%sql
select 'Total Revenue (in INR)'as KPI, FORMAT_NUMBER( sum(Total_revenue_INR), 2) as measure from ecommerce.gold.gld_order_items 
UNION ALL
select 'Gross Sales (before discount)' as KPI, FORMAT_NUMBER(sum(gross_revenue_INR),2) as measure from ecommerce.gold.gld_order_items
union all
SELECT 'Tax Contribution' AS KPI, FORMAT_NUMBER(SUM(tax_amount), 2) AS measure
FROM ecommerce.gold.gld_order_items
UNION ALL
SELECT 'Total Discount' AS KPI, FORMAT_NUMBER(SUM(quantity*unit_price*inr_rate*discount_pct), 2) AS measure
FROM ecommerce.gold.gld_order_items
UNION ALL
SELECT 'Quantity Sold' AS KPI, FORMAT_NUMBER(SUM(quantity), 0) AS measure
FROM ecommerce.gold.gld_order_items
UNION ALL
SELECT 'Total_registered_products' AS KPI, FORMAT_NUMBER(COUNT(product_id), 0) AS measure
FROM ecommerce.gold.gld_products
UNION ALL
SELECT 'Actual_product_sold' AS KPI, FORMAT_NUMBER(COUNT(DISTINCT product_id), 0) AS measure
FROM ecommerce.gold.gld_order_items
UNION ALL
SELECT 'Registered_customers' AS KPI, FORMAT_NUMBER(COUNT(customer_id), 0) AS measure
FROM ecommerce.gold.gld_customers
UNION ALL
SELECT 'Active_customers' AS KPI, FORMAT_NUMBER(COUNT(DISTINCT customer_id), 0) AS measure
FROM ecommerce.gold.gld_order_items;


Multi_variant_analysis: [Product] and [orders] table

Target Variable: Total_revenue

In [0]:
%sql
select * from ecommerce.gold.gld_products
limit 5

In [0]:
%sql
select max('product_id') as Attributes ,count(product_id) as count_value from ecommerce.gold.gld_products
union all
select max('Total_Sku') as Attributes ,count(distinct sku) as count_value from ecommerce.gold.gld_products
union all
select max('Total_unique_brand') as Attributes ,count(distinct brand_code) as count_value from ecommerce.gold.gld_products
union all
select max('Total_unique_category') as Attributes ,count(distinct category_code) as count_value from ecommerce.gold.gld_products
union all
select max('unique_product_color') as Attributes ,count(distinct color) as count_value from ecommerce.gold.gld_products
union all
select max('distinct_size') as Attributes ,count(distinct size) as count_value from ecommerce.gold.gld_products
union all
select max('material_used') as Attributes ,count(distinct material) as count_value from ecommerce.gold.gld_products



In [0]:
%sql
select distinct brand_name, category_name
from ecommerce.gold.gld_products
order by category_name

In [0]:
%sql

SELECT category_name,CONCAT(ROUND(SUM(Total_revenue_INR) / 10000000, 1),' M') AS revenue
FROM ecommerce.gold.gld_order_items AS o
LEFT JOIN ecommerce.gold.gld_products AS p ON p.product_id = o.product_id
GROUP BY category_name
ORDER BY SUM(Total_revenue_INR) DESC;

In [0]:
%sql

-- ZCalculate top 3 brands for each categories

with product_brand_rnk_info as   -- calculate brand rank within each category
(
SELECT category_name,brand_name,
sum(Total_revenue_INR) as total_revenue,
CONCAT(ROUND(SUM(Total_revenue_INR) / 10000000, 1),' M') AS Total_revenue_formated,
row_number()over(partition by category_name order by sum(Total_revenue_INR)desc) as brand_rnk
FROM ecommerce.gold.gld_order_items AS o
LEFT JOIN ecommerce.gold.gld_products AS p ON p.product_id = o.product_id
GROUP BY category_name, brand_name
),

category_rnk_info as             -- calculate category rank
(
SELECT category_name,CONCAT(ROUND(SUM(Total_revenue_INR) / 10000000, 1),' M') AS revenue,
row_number()over(order by SUM(Total_revenue_INR) desc) as category_rnk
FROM ecommerce.gold.gld_order_items AS o
LEFT JOIN ecommerce.gold.gld_products AS p ON p.product_id = o.product_id
GROUP BY category_name
)

                                 -- merge two table [brand and category] and rank top 3 products for each category  
select category_rnk,pb.category_name,brand_name,total_revenue_formated,brand_rnk
from product_brand_rnk_info as pb
left join category_rnk_info as c on pb.category_name = c.category_name
where brand_rnk <= 3
order by category_rnk   



In [0]:
%sql

-- Slice the total_revenue by color. Distribution is almost equal.

-- select distinct color from ecommerce.gold.gld_products
-- output: 13 distinct color

select color, CONCAT(ROUND(SUM(Total_revenue_INR) / 10000000, 1),' M') as Total_revenue_INR
from ecommerce.gold.gld_order_items as o
left join ecommerce.gold.gld_products as p on p.product_id = o.product_id
group by color
order by Total_revenue_INR desc


In [0]:
%sql
-- Slice the total_revenue by color. Distribution is almost equal.

select material, CONCAT(ROUND(SUM(Total_revenue_INR) / 10000000, 1),' M') as Total_revenue_INR
from ecommerce.gold.gld_order_items as o
left join ecommerce.gold.gld_products as p on p.product_id = o.product_id
group by material
order by Total_revenue_INR desc

Multi_variant_analysis: [Customers] and [orders] table

In [0]:
%sql 
select max('Total_registered_customers')as attributes ,count(*) as count_value from ecommerce.gold.gld_customers
union all
select max('Total_unique_country')as attributes ,count(distinct country) as count_value from ecommerce.gold.gld_customers
union all
select max('Total_unique_state')as attributes ,count(distinct state_code) as count_value from ecommerce.gold.gld_customers
union all
select max('Total_unique_region')as attributes ,count(distinct regions) as count_value from ecommerce.gold.gld_customers


In [0]:
%sql

-- Distribution of revenue by countries

select coalesce(country , 'not_available' ) as country, 
CONCAT(ROUND(SUM(Total_revenue_INR) / 10000000, 1),' M') as Total_revenues_INR
-- not_available represent there are few customers whose id is not present in the dim_customer_table and break ref_int, 
-- thats why it produce null.

from ecommerce.gold.gld_order_items as o 
left join ecommerce.gold.gld_customers as c on c.customer_id = o.customer_id
group by country
order by Total_revenues_INR desc;


In [0]:
%sql
select sum(Total_revenue_INR)as total_revenue, CONCAT(ROUND(SUM(Total_revenue_INR) / 10000000, 1),' M')  as formated_total_revenue
from ecommerce.gold.gld_order_items
where cust_id_integrity_flag = 'invalid'

In [0]:
%sql

-- Distribution of revenue by regions

select coalesce(regions,'not_available'), CONCAT(ROUND(SUM(Total_revenue_INR) / 10000000, 1),' M') as Total_revenues_INR
from ecommerce.gold.gld_order_items as o 
left join ecommerce.gold.gld_customers as c on c.customer_id = o.customer_id
group by regions
order by Total_revenues_INR desc

In [0]:
%sql
with top_regions_rank as
(
select regions,sum(Total_revenue_INR)as total_revenue,
row_number()over(order by sum(Total_revenue_INR) desc) as top_rnk
from ecommerce.gold.gld_denormalised_order_items
group by regions
),
bottom_regions_rank as
(
select regions,sum(Total_revenue_INR)as total_revenue,
row_number()over(order by sum(Total_revenue_INR) asc) as bottom_rnk
from ecommerce.gold.gld_denormalised_order_items
group by regions
)

select regions,
format_number(total_revenue/10000000,2) as revenue_in_million,
'Top_4_region' as status
from top_regions_rank
where top_rnk<=4

union all

select coalesce(regions,'not_available'),
format_number(total_revenue/10000000,2) as revenue_in_million,
'Bottom_4_region' as status
from bottom_regions_rank
where bottom_rnk<=4
order by revenue_in_million desc;

Adhoc and Churn Analysis of customers

In [0]:

%sql
with customer_first_and_prev_orders as
(
select *,
LAG(first_month_orders)over(partition by customer_id order by years,months) as prev_month_date
from(
select YEAR(dates)as years,MONTH(dates) as months ,customer_id, 
min(dates) as first_month_orders
from ecommerce.gold.gld_order_items
group by YEAR(dates),MONTH(dates) ,customer_id
)A
)  --select * from customer_first_and_prev_orders order by yr_month
, 
final_tab as
(
select years,months, 
COUNT(*) as total_customers,
sum(case when prev_month_date is null then 1 else 0 end )as new_cutomers,
COUNT(*)-sum(case when prev_month_date is null then 1 else 0 end ) as retained_customers,
sum(case when DATEDIFF(MONTH,prev_month_date,first_month_orders)>1 then 1 else 0 end )as prev_month_churn_cnt
from customer_first_and_prev_orders
group by years,months
)
select years,months,total_customers,new_cutomers,retained_customers,
coalesce(LEAD(prev_month_churn_cnt)over(order by years, months),0)as churn_cnt,
round(
 (coalesce(LEAD(prev_month_churn_cnt)over(order by years, months),0)/total_customers)*100 ,2)as churn_rate_pct
from final_tab
order by years,months

